
---

# Machine Learning Pipelines & Data Leakage

## 1. What is a Pipeline?

In Scikit-Learn, a **Pipeline** binds your preprocessing steps (e.g., missing value imputation, scaling, mathematical transformations) and your final machine learning model into a single, cohesive executable object.

Instead of calling `.fit()` and `.transform()` manually across different scripts for training and testing data, you pass raw data into the pipeline, and it handles the internal sequence automatically.

---

## 2. Core Problems Pipelines Solve

### 1. Data Leakage (The "Future Sight" Problem)

* **The Problem:** Data leakage occurs when information from outside the training dataset is mistakenly used to train the model. For example, if you calculate the overall `mean` of a column across the *entire* dataset before splitting it into train/test sets, your training set implicitly "knows" information about the test set's distribution.
* **The Consequence:** Your validation scores will look phenomenally high during testing, but your model will fall apart and underperform dramatically when deployed in production on truly unseen data.
* **The Pipeline Solution:** A pipeline guarantees that preprocessing parameters (like means, variances, or min/max values) are calculated **only** from the training splits (`X_train`) during `.fit()` and are simply applied to the test sets (`X_test`) without recalculating them.

### 2. Code Duplication & Maintenance Failure

* **The Problem:** Without pipelines, you are forced to repeat code chunks multiple times: once for `X_train`, once for `X_test`, and again later inside a server script when testing an isolated production data point.
* **The Consequence:** If you change an early step (e.g., switching from a Log transform to a Box-Cox transform), you must manually hunt through your scripts to update it everywhere. Missing a single spot causes a data format mismatch, crashing your system or corrupting predictions.
* **The Pipeline Solution:** You modify the transformer element in **one single line** inside the pipeline setup, and both your training, testing, and production code bases update simultaneously.

### 3. The Index Shifting Nightmare

* **The Problem:** Modifying different column types requires Scikit-Learn's `ColumnTransformer`. As discovered on Day 30, this unpacks your data, alters specified columns, stacks them at the front of a raw NumPy array, and drops pandas column names entirely.
* **The Consequence:** Writing custom scripts to track array offsets manually (`[:, 0]`, `[:, 1]`) is prone to human error, making it easy to feed wrong indices into downstream steps.
* **The Pipeline Solution:** Pipelines absorb the raw matrix outputs internally. You don't have to manually slice or track index adjustments between transformer steps because the pipeline passes the correctly re-ordered structural matrix right into the model's `.fit()` function behind the scenes.

---

## 3. The Cross-Validation Impossibility

When running Cross-Validation (`cross_val_score`), Scikit-Learn automatically slices your data into multiple training and validation folds.

* **Why manual preprocessing breaks CV:** If you pre-transform your dataset completely before passing it to `cross_val_score`, the validation fold data was already used to compute global trends during scaling/imputation. This compromises the scientific integrity of cross-validation.
* **The Pipeline Solution:** When you pass a `Pipeline` object directly into `cross_val_score`, Scikit-Learn strictly isolates preprocessing to occur *inside* each individual cross-validation split cycle.

```python
from sklearn.pipeline import Pipeline

# 1. Bundle preprocessing and modeling together
pipeline = Pipeline([
    ('preprocessor', my_column_transformer),
    ('model', LogisticRegression())
])

# 2. Preprocessing is now safely isolated inside each validation fold
scores = cross_val_score(pipeline, X, y, cv=10, scoring='accuracy')

```

---

## 4. Pipeline Execution Workflow Blueprint

| When you run... | What the Pipeline does under the hood |
| :--- | :--- |
| **`pipeline.fit(X_train, y_train)`** | 1. Calls `.fit_transform()` on the first transformer step.<br>2. Passes that transformed data to the next step.<br>3. Continues down the chain until it reaches the final model, where it calls `.fit()`. |
| **`pipeline.predict(X_test)`** | 1. Calls `.transform()` on your test data using the parameters it *already* learned during training.<br>2. Passes the preprocessed test array directly to the model to output final predictions. **(It never refits data during prediction!)** |